# PROTIUM Decay Workbook
### PROTIUM framework — nuclear decay examples in QUIP / CHIP / BLIP units

This notebook demonstrates nuclear decay Q-values and half-lives expressed in the native units of the **PROTIUM framework**, derived from the hydrogen 21 cm hyperfine transition.

| Unit | Definition | SI equivalent |
|------|-----------|---------------|
| **BLIP** | 10¹⁰ × τ(H) | ≈ 7.04 s |
| **QUIP** | 10¹⁰ × E(H) | ≈ 58.74 keV (≈ 9.41 × 10⁻¹⁵ J) |
| **CHIP** | 10¹⁰ × mₚ | ≈ 16.73 fg (≈ 1.67 × 10⁻¹⁷ kg) |

where τ(H) = 1/ν(H), E(H) = hν(H), and ν(H) = 1,420,405,751.768 Hz is the hydrogen hyperfine frequency. All named units are exactly 10¹⁰ × the corresponding base unit.

---
**Data source:** `protium_decays.json` — expandable lookup table.  
Add new entries to that file; this notebook recalculates PROTIUM values automatically.

In [ ]:

# === PROTIUM Constants ===
import sys, os

# Try to import from the PROTIUM library (available when running inside the repo).
# Falls back to inline values if the package is not installed, so the notebook
# works as a standalone export without the source tree.
try:
    _repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if _repo_root not in sys.path:
        sys.path.insert(0, _repo_root)
    from protium import H_FREQ, H_PLANCK, C, BLIP, QUIP, CHIP, M_PROTON, E_H
    _source = 'PROTIUM library'
except ImportError:
    # Inline fallback — exact values, SI 2019 / IAU 2015
    H_FREQ    = 1_420_405_751.768   # Hz   — hydrogen hyperfine frequency
    H_PLANCK  = 6.62607015e-34      # J·s  — Planck constant (exact, SI 2019)
    C         = 299_792_458.0       # m/s  — speed of light (exact, SI 2019)
    BLIP      = 1e10 / H_FREQ      # s    — 10¹⁰ × τ(H)
    E_H       = H_PLANCK * H_FREQ   # J    — base quantum He☉
    QUIP      = 1e10 * E_H          # J    — 10¹⁰ × E(H) ≈ 58.74 keV
    M_PROTON  = 1.67262192595e-27   # kg   — proton mass (CODATA 2018)
    CHIP      = 1e10 * M_PROTON     # kg   — 10¹⁰ × mₚ ≈ 16.73 fg
    _source = 'inline fallback (PROTIUM not installed)'

# Convenience aliases and derived units
f_H     = H_FREQ
h       = H_PLANCK
c       = C
eV      = 1.602176634e-19     # J   — electron volt (exact, SI 2019)
MeV     = eV * 1e6
keV     = eV * 1e3
fg      = 1e-18               # kg  — one femtogram

BLIP_s  = BLIP
QUIP_J  = QUIP
CHIP_kg = CHIP

print(f'PROTIUM Named Units [{_source}]')
print(f'  1 BLIP = {BLIP_s:.6f} s')
print(f'  1 QUIP = {QUIP_J:.6e} J  ({QUIP_J/keV:.2f} keV)')
print(f'  1 CHIP = {CHIP_kg:.6e} kg  ({CHIP_kg/fg:.2f} fg)')
print()
print('Scaling check:')
print(f'  QUIP / E(H) = {QUIP_J / E_H:.0e}  (10¹⁰ by definition)')
print(f'  CHIP / mₚ   = {CHIP_kg / M_PROTON:.0e}  (10¹⁰ by definition)')


In [ ]:
import json, math, os, sys

# Locate protium_decays.json. Three paths in priority order:
#   1. CWD/protium_decays.json        — Jupyter launched from notebook/
#   2. CWD/notebook/protium_decays.json — Jupyter launched from repo root
#   3. HTTP fetch from raw GitHub   — Google Colab (no local filesystem)
_GITHUB_RAW = (
    'https://raw.githubusercontent.com/patrixmyth/protium/main/protium_decays.json'
)

_candidates = [
    os.path.join(os.getcwd(), 'protium_decays.json'),
    os.path.join(os.getcwd(), 'notebook', 'protium_decays.json'),
]
_data_path = next((p for p in _candidates if os.path.isfile(p)), None)

if _data_path is not None:
    with open(_data_path) as _f:
        DECAY_DATA = json.load(_f)
    _source_label = os.path.basename(_data_path)
else:
    # Colab / offline export: fetch from GitHub
    try:
        import urllib.request
        with urllib.request.urlopen(_GITHUB_RAW) as _resp:
            DECAY_DATA = json.loads(_resp.read().decode())
        _source_label = 'GitHub (remote fetch)'
    except Exception as _e:
        raise FileNotFoundError(
            'protium_decays.json not found locally and remote fetch failed. '
            f'Error: {_e}\n'
            'Run Jupyter from notebook/ or the repo root, or check network access.'
        ) from _e

decays = DECAY_DATA['decays']

# Recompute PROTIUM values from source-of-truth fields (Q_MeV, halflife_s).
# eV and MeV must be defined before this cell runs (see constants cell).
for d in decays:
    Q_J = d['Q_MeV'] * MeV
    d['Q_quips']            = Q_J / QUIP_J
    d['Q_quips_exp']        = math.log10(d['Q_quips'])
    d['delta_m_chips']      = (Q_J / c**2) / CHIP_kg   # Δm = Q/c², in CHIPs
    d['delta_m_chips_exp']  = math.log10(d['delta_m_chips'])
    hl = d['halflife_s']
    d['halflife_blips']     = hl / BLIP_s
    d['halflife_blips_exp'] = math.log10(hl / BLIP_s) if hl > 0 else None

print(f'Loaded {len(decays)} decay entries from {_source_label}')
for d in decays:
    print(f"  {d['parent']} -> {d['daughter']}  [{d['mode']}]  Q={d['Q_MeV']} MeV  t1/2={d['halflife_human']}")

---
## Worked Example: Tritium Decay

Tritium (H-3) is the most self-referential decay in the PROTIUM framework: hydrogen defines the unit system, and here hydrogen *decays*.

$$\mathrm{^3H} \rightarrow \mathrm{^3He} + e^- + \bar{\nu}_e \qquad Q = 18.591\,\mathrm{keV}$$

We express Q in **QUIPs**, mass deficit in **CHIPs**, and half-life in **BLIPs**.

In [ ]:

def worked_example(decay_id):
    """Full PROTIUM worked example for a decay entry by id."""
    match = [x for x in decays if x['id'] == decay_id]
    if not match:
        valid = ', '.join(d['id'] for d in decays)
        print(f"Unknown decay id: '{decay_id}'")
        print(f"Available ids: {valid}")
        return
    d = match[0]
    sep = '=' * 60
    print(sep)
    print(f"  {d['name']}")
    print(f"  {d['parent']} -> {d['daughter']}  [{d['mode']}]")
    print(sep)
    print()

    Q_J = d['Q_MeV'] * MeV
    print('Q-value (SI):')
    print(f"  Q = {d['Q_MeV']} MeV")
    print(f"    = {d['Q_MeV']} × 10⁶ × eV                  [1 MeV = 10⁶ eV]")
    print(f"    = {d['Q_MeV']} × 10⁶ × {eV:.6e} J/eV    [eV exact, SI 2019]")
    print(f"    = {Q_J:.4e} J")
    print()

    print('Q-value in QUIPs  [1 QUIP = 10¹⁰ × E(H) ≈ 58.74 keV]:')
    print(f"  Q = {Q_J:.4e} J ÷ {QUIP_J:.4e} J/QUIP")
    print(f"    = {d['Q_quips']:.4f} QUIPs")
    print(f"  log₁₀(Q/QUIP) = {d['Q_quips_exp']:.4f}")
    print()

    dm_kg = Q_J / c**2
    print('Mass deficit in CHIPs  [1 CHIP = 10¹⁰ × mₚ ≈ 16.73 fg]:')
    print(f"  Δm = Q / c²")
    print(f"     = {Q_J:.4e} J ÷ ({c:.6e} m/s)²")
    print(f"     = {dm_kg:.4e} kg")
    print(f"     = {d['delta_m_chips']:.4e} CHIPs")
    print(f"  log₁₀(Δm/CHIP) = {d['delta_m_chips_exp']:.4f}")
    print()

    print('Half-life in BLIPs  [1 BLIP = 10¹⁰ × τ(H) ≈ 7.04 s]:')
    print(f"  t½ = {d['halflife_human']} = {d['halflife_s']:.4e} s")
    print(f"     = {d['halflife_s']:.4e} s ÷ {BLIP_s:.6f} s/BLIP")
    print(f"     = {d['halflife_blips']:.4e} BLIPs")
    if d['halflife_blips_exp'] is not None:
        nt = round(d['halflife_blips_exp'])
        dt = d['halflife_blips_exp'] - nt
        print(f"  log₁₀(t½/BLIP) = {d['halflife_blips_exp']:.4f}")
        print(f"  ≈ 10^{nt}  (Δ = {dt:+.4f} decades)")

    if d.get('notes'):
        print()
        print(f"Notes: {d['notes']}")
    print()

if 'decays' in vars():
    worked_example('H3-He3')
else:
    print('[Run from the top — Runtime > Run all cells]')


---
## Decay Explorer

Browse all table entries. Adjust `filter_mode` and `sort_by` to explore.

In [ ]:
filter_mode = None           # None = all, or 'alpha' 'beta-' 'beta+' 'gamma'
sort_by     = 'Q_quips_exp'  # 'Q_quips_exp' | 'halflife_blips_exp' | 'name'

subset = [d for d in decays if filter_mode is None or d['mode'] == filter_mode]
subset = sorted(subset, key=lambda x: x.get(sort_by) or 0)

hdr = f"{'Decay':<30} {'Mode':<8} {'Q (MeV)':<12} {'log10(Q/QUIP)':<16} {'t1/2':<14} {'log10(t1/2/BLIP)'}"
print(hdr)
print('-' * len(hdr))
for d in subset:
    texp = f"{d['halflife_blips_exp']:.3f}" if d['halflife_blips_exp'] else '---'
    print(f"{d['parent']+' -> '+d['daughter']:<30}{d['mode']:<8}{d['Q_MeV']:<12}{d['Q_quips_exp']:<16.4f}{d['halflife_human']:<14}{texp}")

In [ ]:
# Change the id to any entry in the table above
worked_example('Bi212-Tl208')

---
## PROTIUM Log-Scale Position

Each decay plotted on the QUIP (energy) and BLIP (time) log axes.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

colors = {'alpha': '#e05c2a', 'beta-': '#2a7ae0', 'beta+': '#2ae07a', 'gamma': '#9b2ae0'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Nuclear Decays on the PROTIUM Log Scale', fontsize=14, fontweight='bold')

for ax, key, label in [
    (axes[0], 'Q_quips_exp',        'log₁₀(Q / QUIP)'),
    (axes[1], 'halflife_blips_exp',  'log₁₀(t½ / BLIP)'),
]:
    # Collect points with values, sorted by x for consistent stagger
    pts = [(d, d.get(key)) for d in decays if d.get(key) is not None]
    pts.sort(key=lambda p: p[1])

    for i, (d, val) in enumerate(pts):
        col = colors.get(d['mode'], 'grey')
        # Alternate labels above and below the axis
        y_off = 14 if i % 2 == 0 else -18
        va = 'bottom' if i % 2 == 0 else 'top'
        ax.scatter(val, 0, color=col, s=120, zorder=3)
        ax.annotate(d['parent'], (val, 0),
                    textcoords='offset points', xytext=(0, y_off),
                    ha='center', va=va, fontsize=8, rotation=45)

    ax.set_xlabel(label, fontsize=11)
    ax.set_yticks([])
    ax.grid(axis='x', alpha=0.3)

axes[0].axvline(0, color='gold', linestyle='--', linewidth=1.5, label='1 QUIP ≈ 58.74 keV')
axes[0].set_title('Q-value (QUIPs)')
axes[0].legend(fontsize=8)
axes[1].set_title('Half-life (BLIPs)')

from matplotlib.lines import Line2D
leg = [Line2D([0],[0], marker='o', color='w', markerfacecolor=v, markersize=9, label=k)
       for k, v in colors.items()]
fig.legend(handles=leg, loc='lower center', ncol=4, fontsize=9, title='Decay mode')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


---
## Adding New Entries

Add a JSON object to the `decays` array in `protium_decays.json`:

```json
{
  "id":             "unique-string",
  "name":           "Human-readable name",
  "parent":         "Po-210",
  "daughter":       "Pb-206",
  "mode":           "alpha",
  "Q_MeV":          5.3044,
  "halflife_s":     11955168,
  "halflife_human": "138.4 days",
  "notes":          "optional context"
}
```

The notebook recomputes all PROTIUM quantities from `Q_MeV` and `halflife_s` — no manual calculation needed.

**Data sources:**
- [NUBASE2020](https://www.sciencedirect.com/science/article/pii/S1674198120001458)
- [NNDC Chart of Nuclides](https://www.nndc.bnl.gov/nudat3/)
- [IAEA Live Chart](https://www-nds.iaea.org/relnsd/vcharthtml/VChartHTML.html)